# pinball — DDQN on Pinball `easy`, buffer-size comparison

Compares the `large` component (buffer 10,000) against each smaller-buffer
variant, `small-10%` (buffer 1,000) and `small-5%` (buffer 500), as episodic-
return curves with mean ± 95% bootstrap-CI bands over 30 seeds each, loaded
from `results/{large,small-10%,small-5%}.db`.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from experiment import load_curves, load_runs
from experiment.plotting import seed_grids_for, plot_mean_ci, style


def grid_for(component_db, n=500):
    """A shared [0, T] timestep grid from any run's curve length (T = TOTAL_TIMESTEPS)."""
    df = load_runs(component_db, write_csv=False)
    T = len(load_curves(component_db, df["run_id"][0])["reward"])
    return np.linspace(0, T, n)

In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else Path("experiments/pinball/results")

GRID = grid_for(RESULTS / "large.db")

COLORS = {
    "large": "tab:blue",
    "small-10%": "tab:orange",
    "small-5%": "tab:red",
}


def plot_comparison(names, title):
    """Draw one figure: mean ± 95% bootstrap CI for each named component."""
    fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
    for name in names:
        stack = seed_grids_for(RESULTS / f"{name}.db", GRID)
        plot_mean_ci(ax, GRID, stack, f"{name} (n={stack.shape[0]})", COLORS[name])
    ax.set_title(f"{title}  (mean ± 95% bootstrap CI)")
    ax.legend(loc="lower right", frameon=False)
    style(ax, ylim=(-1000, 0))
    fig.tight_layout()
    plt.show()
    return fig

In [ ]:
fig_10 = plot_comparison(
    ["large", "small-10%"], "DDQN on Pinball easy — buffer 10,000 vs 1,000"
)

In [ ]:
fig_5 = plot_comparison(
    ["large", "small-5%"], "DDQN on Pinball easy — buffer 10,000 vs 500"
)

In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
fig_10.savefig(PLOTS_DIR / "pinball_large_vs_small10.pdf", bbox_inches="tight")
fig_5.savefig(PLOTS_DIR / "pinball_large_vs_small5.pdf", bbox_inches="tight")
print(f"saved plots to {PLOTS_DIR}")